In [1]:
from pathlib import Path
import json
import pickle
import numpy as np
from collections import deque
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from groq import Groq

c:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from io import BytesIO

from barcode import EAN13
from barcode.writer import SVGWriter

# Write to a file-like object:
rv = BytesIO()
EAN13("100000902922", writer=SVGWriter()).write(rv)

# Or to an actual file:
with open("somefile.svg", "wb") as f:
    EAN13(str(100000011111), writer=SVGWriter()).write(f)

In [18]:
# Generate Code39 barcodes and save as SVG files
from io import BytesIO
from barcode import Code39
from barcode.writer import SVGWriter
x = input(" ").split(",")
for barcode in x:
    code = Code39(barcode)
    code.save(f"{barcode}")

In [ ]:
CHUNKS_PATH = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\unified_semantic_chunks\unified_chunks.json"
)
VECTOR_STORE = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\vector_store"
)
CONFIG_PATH   = VECTOR_STORE / "config.json"
FAISS_PATH    = VECTOR_STORE / "faiss.index"
METADATA_PATH = VECTOR_STORE / "metadata.pkl"

# Retrieval settings
VECTOR_CANDIDATES = 40
RERANK_CANDIDATES = 25
TOP_K_DEFAULT     = 5

# Score weights
W_VECTOR  = 0.6
W_BM25    = 0.3
W_HYBRID  = 0.7
W_RERANK  = 0.3


CONFIDENCE_THRESHOLD = 0.10


GROQ_API_KEY = "gsk_REDACTED_KEY_WAS_ROTATED"
GROQ_MODEL   = "openai/gpt-oss-120b"   

print("✅ Config set")

In [ ]:
if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)
    MODEL_NAME = config["model_name"]
    print(f"✅ Config loaded — model: {MODEL_NAME}")
else:
    MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
    print(f"⚠️  No config found — using default: {MODEL_NAME}")

In [ ]:
# Raw chunks — used for BM25
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)
chunks = [c for c in all_chunks if c.get("text", "").strip()]
print(f"✅ Raw chunks loaded: {len(chunks)}")

# Enriched/boosted chunks — aligned with FAISS index
with open(METADATA_PATH, "rb") as f:
    enriched_chunks = pickle.load(f)
print(f"✅ Enriched chunks loaded: {len(enriched_chunks)}")

# FAISS index
index = faiss.read_index(str(FAISS_PATH))
print(f"✅ FAISS index loaded — {index.ntotal} vectors, dim={index.d}")

In [ ]:
print(f"🔄 Loading embedding model: {MODEL_NAME}")
embedding_model = SentenceTransformer(MODEL_NAME)

print("🔄 Loading reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("✅ Models ready")

In [ ]:
# Built on raw chunks only — avoids keyword inflation from boosted duplicates
tokenized_corpus = [c["text"].lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)
print(f"✅ BM25 built on {len(tokenized_corpus)} chunks")

In [ ]:
def detect_document_type(chunk: dict) -> str:
    """Uses chunk_type metadata first, falls back to structured_data inspection."""
    chunk_type = chunk.get("metadata", {}).get("chunk_type", "")

    if chunk_type in ("schema_overview", "schema_core_columns", "schema_extra_columns"):
        return "TABLE_SCHEMA"
    if chunk_type in ("wms_overview", "wms_join_logic", "wms_procedure", "wms_safety_rules"):
        return "OPERATIONAL_REFERENCE"
    if chunk_type in ("text_prose", "text_table"):
        return "TEXT"

    # Fallback
    structured = chunk.get("structured_data")
    if isinstance(structured, dict):
        if "columns" in structured:
            return "TABLE_SCHEMA"
        if "procedures" in structured or "core_tables" in structured:
            return "OPERATIONAL_REFERENCE"
    return "TEXT"

In [ ]:
SCHEMA_KEYWORDS = {
    "sql", "select", "query", "join", "where", "insert", "update",
    "column", "columns", "table", "schema", "foreign key", "primary key",
    "field", "fields", "datatype", "varchar", "integer", "structure",
    "definition", "describe", "what is the structure"
}

OPERATIONAL_KEYWORDS = {
    "reverse", "reset", "grn", "receipt", "shipment", "mission",
    "cancel", "validate", "close", "reopen", "resend", "loading",
    "inbound", "outbound", "picking", "putaway", "stock", "movement",
    "how do i", "how to", "steps to", "procedure for"
}

def classify_query(query: str) -> dict:
    query_lower      = query.lower()
    schema_hits      = [k for k in SCHEMA_KEYWORDS      if k in query_lower]
    operational_hits = [k for k in OPERATIONAL_KEYWORDS if k in query_lower]

    is_schema      = len(schema_hits) > 0
    is_operational = len(operational_hits) > 0

    # Operational wins when both match (more specific in WMS context)
    if is_schema and is_operational:
        is_schema = False

    return {
        "is_schema"        : is_schema,
        "is_operational"   : is_operational,
        "schema_hits"      : schema_hits,
        "operational_hits" : operational_hits
    }

In [ ]:
def retrieve_context(query: str, top_k: int = TOP_K_DEFAULT, verbose: bool = False):
    query_lower  = query.lower()
    query_tokens = query_lower.split()
    intent       = classify_query(query)

    if verbose:
        print(f"🔍 Intent: schema={intent['is_schema']}, operational={intent['is_operational']}")

    # --- Vector search ---
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, VECTOR_CANDIDATES)

    # --- BM25 ---
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_max    = bm25_scores.max() if bm25_scores.max() > 0 else 1.0
    bm25_norm   = bm25_scores / bm25_max

    # --- Merge ---
    seen_texts = {}

    for vector_score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        chunk     = enriched_chunks[idx]
        metadata  = chunk.get("metadata", {})
        doc_type  = detect_document_type(chunk)
        text      = chunk["text"]

        if text in seen_texts:
            continue

        # Map to raw chunk for BM25 score
        source   = metadata.get("source", "")
        chunk_id = chunk.get("chunk_id", -1)
        raw_idx  = next(
            (i for i, c in enumerate(chunks)
             if c.get("metadata", {}).get("source") == source
             and c.get("chunk_id") == chunk_id),
            None
        )
        bm25_score = float(bm25_norm[raw_idx]) if raw_idx is not None else 0.0

        v_score_norm = (float(vector_score) + 1) / 2
        hybrid = (W_VECTOR * v_score_norm) + (W_BM25 * bm25_score)

        # Intent boosts
        if intent["is_operational"] and doc_type == "OPERATIONAL_REFERENCE":
            hybrid += 0.4
        if intent["is_schema"] and doc_type == "TABLE_SCHEMA":
            hybrid += 0.3

        # Name match boosts
        table_name = metadata.get("table_name", "")
        if table_name and table_name.lower() in query_lower:
            hybrid += 0.5

        proc_name = metadata.get("procedure_name", "")
        if proc_name and proc_name.lower() in query_lower:
            hybrid += 0.4

        for rt in metadata.get("related_tables", []):
            if rt and rt.lower() in query_lower:
                hybrid += 0.2
                break

        seen_texts[text] = {
            "hybrid_score"   : hybrid,
            "vector_score"   : v_score_norm,
            "bm25_score"     : bm25_score,
            "doc_type"       : doc_type,
            "text"           : text,
            "metadata"       : metadata,
            "structured_data": chunk.get("structured_data")
        }

    if not seen_texts:
        return [], 0.0

    results    = sorted(seen_texts.values(), key=lambda x: x["hybrid_score"], reverse=True)
    candidates = results[:RERANK_CANDIDATES]

    # --- Rerank ---
    pairs         = [(query, r["text"]) for r in candidates]
    rerank_scores = reranker.predict(pairs)

    for i, r in enumerate(candidates):
        r["rerank_score"] = float(rerank_scores[i])
        r["final_score"]  = (W_HYBRID * r["hybrid_score"]) + (W_RERANK * r["rerank_score"])

    candidates  = sorted(candidates, key=lambda x: x["final_score"], reverse=True)
    top_results = candidates[:top_k]
    confidence  = float(np.mean([r["final_score"] for r in top_results]))

    return top_results, confidence

In [ ]:
def validate_context(retrieved: list, confidence: float) -> bool:
    """
    Two-gate validation:
    1. Confidence score must exceed threshold
    2. At least one result must exist

    Fix: your original always passed (threshold was 0.00) which
    meant it never actually filtered anything out.
    """
    if not retrieved:
        return False
    if confidence < CONFIDENCE_THRESHOLD:
        return False
    return True

In [ ]:
def build_context_text(retrieved: list) -> tuple:
    """
    Expands structured schema data into full readable column tables.
    Returns (context_text, has_schema, has_operational).
    """
    context_sections = []
    has_schema       = False
    has_operational  = False

    for r in retrieved:
        metadata   = r.get("metadata", {})
        text       = r.get("text", "")
        structured = r.get("structured_data")
        doc_type   = r.get("doc_type", "TEXT")

        if doc_type == "TABLE_SCHEMA":
            has_schema = True

            # Expand full schema from structured_data if available
            if isinstance(structured, dict) and "columns" in structured:
                table_name  = structured.get("table_name", "UNKNOWN")
                description = structured.get("description", "N/A")
                primary_key = structured.get("primary_key", "N/A")
                columns     = structured.get("columns", [])

                lines = [
                    f"TABLE NAME   : {table_name}",
                    f"DESCRIPTION  : {description}",
                    f"PRIMARY KEY  : {primary_key}",
                    f"TOTAL COLUMNS: {len(columns)}",
                    "",
                    "COLUMNS:",
                    f"{'─'*80}"
                ]

                for col in columns:
                    col_lines = [
                        f"  Name        : {col.get('name', 'UNKNOWN')}",
                        f"  Description : {col.get('description', 'N/A')}",
                        f"  SQL Server  : {col.get('type_sql_server', 'N/A')}",
                        f"  Oracle      : {col.get('type_oracle', 'N/A')}",
                        f"  Primary Key : {'Yes' if col.get('is_primary_key') else 'No'}",
                        f"  Foreign Key : {'Yes' if col.get('is_foreign_key') else 'No'}",
                    ]
                    if col.get("references_table"):
                        col_lines.append(
                            f"  References  : {col['references_table']}.{col.get('references_column','?')}"
                        )
                    lines.extend(col_lines)
                    lines.append(f"  {'─'*40}")

                text = "\n".join(lines)

        elif doc_type == "OPERATIONAL_REFERENCE":
            has_operational = True

        context_sections.append(
            f"[SOURCE: {metadata.get('source','unknown')} | "
            f"CATEGORY: {metadata.get('category','unknown')} | "
            f"TYPE: {doc_type} | "
            f"TABLE: {metadata.get('table_name','N/A')}]\n{text}"
        )

    return "\n\n".join(context_sections), has_schema, has_operational


def build_prompt(query: str, retrieved: list, memory_text: str = "") -> str:
    context_text, has_schema, has_operational = build_context_text(retrieved)

    query_lower       = query.lower()
    is_schema_question = any(w in query_lower for w in [
        "column", "columns", "schema", "structure", "fields",
        "table definition", "describe", "what is the structure"
    ])
    is_sql_question = any(w in query_lower for w in [
        "sql", "select", "query", "join", "write a query"
    ])

    # --- Schema mode instructions ---
    schema_hint = ""
    if has_schema and is_schema_question:
        schema_hint = """
SCHEMA MODE ACTIVE:
- Provide the COMPLETE schema definition
- Include: Table Name, Description, Primary Key, Total Columns
- List ALL columns with: Name, Description, SQL Server Type, Oracle Type, PK (Yes/No), FK (Yes/No), References
- Do NOT skip any columns
- Do NOT omit metadata
- If schema is incomplete in context, state that explicitly
"""

    # --- SQL mode instructions ---
    sql_hint = ""
    if is_sql_question:
        sql_hint = """
SQL MODE ACTIVE:
- Generate production-ready SQL only
- Use explicit JOIN conditions based on FK relationships in the context
- Do NOT use SELECT *
- Do NOT invent tables or columns not present in the context
- Use proper indentation and formatting
- Only return SELECT queries unless the user explicitly asks for UPDATE/INSERT
- Base all JOIN logic on the foreign key relationships described in the metadata
"""

    # --- Operational mode instructions ---
    operational_hint = ""
    if has_operational:
        operational_hint = """
OPERATIONAL MODE ACTIVE:
- Follow the exact procedure steps described in the context
- Reference the specific SQL provided if present
- Respect all safety rules listed in the context
- State the access level required for the procedure
"""

    prompt = f"""You are the Technical Architect of Speed WMS.

CRITICAL RULES:
- Answer STRICTLY using the provided context below
- Do NOT invent columns, tables, relationships, or procedures
- Do NOT use general SQL knowledge to fill gaps
- If the answer is not in the context, respond exactly:
  "I do not have enough information to answer this. Please contact support."

======================
CONVERSATION HISTORY
======================
{memory_text if memory_text else "(No prior conversation)"}

======================
RETRIEVED CONTEXT
======================
{context_text}

======================
USER QUESTION
======================
{query}

======================
INSTRUCTIONS
======================
{schema_hint}
{sql_hint}
{operational_hint}

Provide a clear, professional, well-structured response:"""

    return prompt.strip()

In [ ]:
client = Groq(api_key=GROQ_API_KEY)

def get_llm_answer(prompt: str) -> str:
    """
    Fix: 'openai/gpt-oss-120b' is not a valid Groq model.
    Using llama3-70b-8192 which is available on Groq and strong
    enough for SQL generation and schema questions.
    Other valid options: mixtral-8x7b-32768, llama3-8b-8192
    """
    try:
        completion = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role"   : "system",
                    "content": (
                        "You are a Speed WMS expert and Technical Architect. "
                        "You answer strictly from provided context. "
                        "You never invent information."
                    )
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=2048
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ LLM call failed: {e}")
        return "I encountered an error generating a response. Please try again."

In [ ]:
def validate_answer(answer: str) -> bool:
    """
    Fix: your original checked if certain phrases appeared in the answer
    and returned False if they did — but those phrases are CORRECT refusals,
    not hallucinations. This caused valid "I don't know" answers to be
    double-wrapped in a second refusal message.

    Correct logic: only reject genuinely bad answers (too short, error messages).
    """
    if not answer or len(answer.strip()) < 20:
        return False

    # Reject if LLM returned an obvious error
    error_phrases = ["error occurred", "exception", "traceback"]
    if any(p in answer.lower() for p in error_phrases):
        return False

    return True

In [ ]:
class ConversationMemory:
    def __init__(self, max_turns: int = 8):
        self.history = deque(maxlen=max_turns * 2)  # *2 because each turn = user + assistant

    def add_turn(self, question: str, answer: str):
        self.history.append({"role": "user",      "content": question})
        self.history.append({"role": "assistant", "content": answer})

    def format(self) -> str:
        if not self.history:
            return ""
        lines = []
        for m in self.history:
            role = "USER" if m["role"] == "user" else "ASSISTANT"
            # Truncate long assistant answers in memory to save tokens
            content = m["content"]
            if m["role"] == "assistant" and len(content) > 400:
                content = content[:400] + "... [truncated]"
            lines.append(f"{role}: {content}")
        return "\n".join(lines)

    def clear(self):
        self.history.clear()
        print("🗑️  Memory cleared")

memory = ConversationMemory(max_turns=8)
print("✅ Memory initialised")

In [ ]:
def ask(question: str, verbose: bool = False) -> str:
    print(f"\n{'='*70}")
    print(f"❓ Question: {question}")
    print(f"{'='*70}")

    # Step 1: Retrieve
    retrieved, confidence = retrieve_context(question, verbose=verbose)
    print(f"📊 Confidence: {confidence:.4f} | Results: {len(retrieved)}")

    # Step 2: Validate context
    if not validate_context(retrieved, confidence):
        refusal = (
            "I do not have enough information to answer this confidently. "
            "Please contact support."
        )
        print(f"⚠️  Context validation failed (confidence={confidence:.4f} < {CONFIDENCE_THRESHOLD})")
        return refusal

    # Step 3: Build prompt
    prompt = build_prompt(
        query        = question,
        retrieved    = retrieved,
        memory_text  = memory.format()
    )

    if verbose:
        print(f"\n📝 Prompt length: {len(prompt)} chars")

    # Step 4: Get LLM answer
    answer = get_llm_answer(prompt)

    # Step 5: Validate answer
    if not validate_answer(answer):
        return (
            "I was unable to generate a valid response. "
            "Please rephrase your question or contact support."
        )

    # Step 6: Update memory
    memory.add_turn(question, answer)

    return answer

In [ ]:
questions = [
    "What is the structure of the Reception Header table?",
    "How do I reverse a GRN?",
    "Show me the SQL to check loading details for an order",
    "What are the foreign keys in REE_DAT?",
    "Explain the warehouse picking process",
]

for q in questions:
    answer = ask(q)
    print(f"\n--- RESPONSE ---\n{answer}\n")

In [ ]:
# Run this cell on its own for ad-hoc testing
answer = ask("can you please join the stock table(stk_dat) with the third party table(tie_par)", verbose=True)
print(f"\n--- RESPONSE ---\n{answer}")

In [ ]:
memory.clear()